### Discriptive Analysis for car accident

## 原始資料預覽的用途

這份 Notebook 保留最初的讀檔與前幾列檢視，用來熟悉原始欄位與內容。
完整的資料準備、分布摘要與夜間比較已分別放在以下文件：

- [資料整理程式](../code/prepare_descriptive_statistics.ipynb)：來源核對、中文代碼及夜間指標。
- [衍生資料說明](../processed/taipei_2025_descriptive_documentation.ipynb)：欄位定義、數值及七個獨立類別摘要。
- [正式敘述性統計](../../finding/敘述性統計.ipynb)：年度概況、光線分布與夜間照明比較。

### 載入 pandas

`pandas` 用於讀取 CSV 與操作表格。此處先保留最初使用的簡單讀法；
下方相對路徑以本 Notebook 所在的 `data/raw/` 為工作目錄。

In [1]:
import pandas as pd

### 讀取原始 CSV

每個數字欄位不一定都適合計算平均數。例如光線、天候與道路型態是代碼，
需要對照 [事故對照表.pdf](事故對照表.pdf) 才能理解；死亡人數、受傷人數則是計數。
正式分析另外保留精確路徑與檔案雜湊，以確保不同人讀取的是相同資料版本。

In [2]:
df = pd.read_csv("114年-臺北市死傷交通事故明細.csv")

### 檢視前五筆

`df.head()` 預設顯示前五筆，適合確認欄位名稱及內容是否成功讀入。
這五筆依原始資料順序出現，不能據此判斷全年事故的月份、地區或照明分布。

In [3]:
df.head()

,發生年度,發生月,發生日,發生時-Hours,發生分,區序,肇事地點,死亡人數,2-30日死亡人數,受傷人數,...,當事者行動狀態,駕駛資格情形,駕駛執照種類,飲酒情形,車輛撞擊部位1,肇因碼-個別,個人肇逃否,職業,座標-X,座標-Y
0,114,1,1,0,18,03中山區,中山區市民大道2段47號,0.0,0.0,1.0,...,9.0,1.0,10.0,1.0,11.0,16.0,1.0,21.0,121.529004,25.046603
1,114,1,1,0,31,10文山區,文山區新光路1段109號旁,0.0,0.0,3.0,...,1.0,1.0,10.0,1.0,14.0,19.0,1.0,21.0,121.573689,24.991275
2,114,1,1,0,35,03中山區,中山區北安路與大直街口,0.0,0.0,1.0,...,5.0,3.0,18.0,1.0,12.0,96.0,1.0,21.0,121.547196,25.079842
3,114,1,1,0,41,03中山區,中山區民族東路2號,0.0,0.0,1.0,...,1.0,1.0,4.0,1.0,8.0,19.0,1.0,21.0,121.523305,25.068395
4,114,1,1,0,57,11南港區,南港區忠孝東路6段與玉成街口,0.0,0.0,1.0,...,9.0,1.0,8.0,1.0,5.0,108.0,1.0,22.0,121.583227,25.045872


### 閱讀預覽時的注意事項

原始資料同時包含事故層級傷亡總數與第一當事人特徵，因此「受傷人數」不等於第一當事人是否受傷。
`光線` 也不能直接取代日夜分類；正式分析使用事故時間與座標計算日出日落。
本次只增加說明文字，保留這份預覽既有的程式與輸出，沒有重新執行。

## 夜間判定的計算來源與定義

**資料來源與方法。** 本研究沒有將中央氣象署公布的日出日落表匯入資料。
日出與日落時間由 Python 的 **Astral 3.2** 計算：輸入每筆事故的西元日期、
`座標-Y`（緯度）與 `座標-X`（經度），海拔統一設為 0 公尺，使用臺灣時間 UTC+8
（2025 年的 Asia/Taipei）。套件依天文模型推算太陽位置及日出日落，並非現場亮度量測。

**主要指標 `is_night`。** 令事故的當地時間為 t、該日該位置的日出與日落為 R、S：
當 **t < R 或 t > S** 時設為 1，其餘為 0；恰好等於日出或日落歸為 0。
若經度或緯度缺失，日出日落及主要夜間指標都保留缺值，不用市中心座標代替。
本資料有 2 筆這類紀錄。`光線` 是照明設備狀態，不能直接取代這個日夜判斷。

**固定指標 `night_fixed`。** 依本次確認的研究設計，採「全年最晚日落之後、
全年最早日出之前」作為保守的固定時段。先對 **2025 年完整 365 日**，
在資料全部有效座標的經緯度外接矩形四角計算日出日落，涵蓋本資料中的位置差異；
經度範圍為 121.4611803–121.6520178、緯度為 24.9659305–25.1832948。
這是本資料的空間範圍，並非臺北市官方行政邊界。

四角全年計算的最早日出為 **2025-06-09 05:02:50.800739**（東北角），
最晚日落為 **2025-07-02 18:48:12.258035**（西北角）。配合事故時間只有分鐘精度，
將最早日出向下取整為 05:02、最晚日落向上取整為 18:49。
故 `night_fixed = 1` 的時段為 **18:49（含）至翌日 05:02（不含）**，
即一天中的分鐘數 m ≥ 1129 或 m < 302，其餘設為 0。
另外逐筆驗證：所有有有效座標且被固定時段納入的事故，必須同時符合其當日 `is_night = 1`。

這個固定指標是較窄的夜間子樣本；值為 0 只表示不在固定時段內，**不代表白天**。
固定時段依時間即可套用；但缺座標者無法逐筆驗證實際位置，所以「皆為夜間」的檢查
限定於有有效座標的紀錄，不能延伸到其他年度、其他地區或未知位置。

**為何不只採夏至？** 夏至白晝最長，並不保證當天是全年日出最早或日落最晚。
此點參考 [臺北市立天文科學教育館的 2025 夏至說明](https://tam.gov.taipei/News_Content.aspx?n=B64052C7930D4913&s=C881A8681CC67490&sms=2CF1F5E2E0B96411)。
該網頁用於確認天文概念；本資料各筆時間與固定界線仍由 Astral 計算，不冒稱為官方公布值。

**解讀限制。** 日落後與日出前可能仍有暮光或曙光；天候、山勢、建築遮蔽與實際照明
也會影響亮度。因此本定義表示模型下的日出日落夜間，不能稱為「完全黑暗」。